# Supply Chain with Delivery Delay

*System Dynamics Studio, model library: Complex behavior*

A retailer orders to replace sales and to correct its inventory, but orders take weeks to arrive. Ignoring the orders already in the pipeline (the supply line) makes the manager over-order, so inventory overshoots and oscillates. This is the structure behind the bullwhip effect.

**Concepts:** Delay, Oscillation, Supply line, Bullwhip

**Expected behavior:** Ignoring the supply line (weight 0) makes orders overshoot: inventory overshoots its target and oscillates before settling.

**Limitations:** Single echelon, constant demand, delivery delay modelled as a first-order (exponential) delay.

## Model structure

**Stock balances**

$$
\begin{aligned}
\frac{d\,\text{Supply\_Line}}{dt} &= \text{orders} - \text{deliveries} \\
\frac{d\,\text{Inventory}}{dt} &= \text{deliveries} - \text{sales} \\
\end{aligned}
$$

**Stocks**

| Stock | Initial value | Unit |
|---|---|---|
| `Supply_Line` | 400 | units |
| `Inventory` | 200 | units |

**Flows and auxiliaries** (in evaluation order)

$$
\begin{aligned}
\text{desired\_supply\_line} &= \text{Customer\_Demand}\cdot\text{delivery\_delay} \\
\text{orders} &= \operatorname{max}\left(0, \text{Customer\_Demand}+\frac{\left(\text{Target\_Inventory}-\text{Inventory}\right)}{\text{inventory\_adjustment\_time}}+\frac{\text{supply\_line\_weight}\cdot\left(\text{desired\_supply\_line}-\text{Supply\_Line}\right)}{\text{inventory\_adjustment\_time}}\right) \\
\text{deliveries} &= \frac{\text{Supply\_Line}}{\text{delivery\_delay}} \\
\text{sales} &= \text{Customer\_Demand} \\
\end{aligned}
$$

**Parameters**

| Parameter | Value | Unit |
|---|---|---|
| `Customer_Demand` | 100 | units/week |
| `Target_Inventory` | 400 | units |
| `inventory_adjustment_time` | 2 | week |
| `delivery_delay` | 4 | week |
| `supply_line_weight` | 0 | dimensionless |

## Setup

NumPy, SciPy and Matplotlib are preinstalled on Google Colab. The step, ramp and pulse helpers
are the time functions available in the app; they are defined here for completeness.

In [ ]:
# Generated by System Dynamics Studio
# Model time: 0 to 80 week, output every 0.1
# In-app solver: RK4, fixed dt = 0.1.
# This script integrates with scipy.integrate.odeint (adaptive LSODA), so values
# match the app within numerical tolerance, not digit for digit.
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# Time functions (discontinuous; same definitions as in the app)
def sd_step(height, start, t):
    return np.where(t >= start, height, 0.0)

def sd_ramp(slope, start, *rest):
    *end, t = rest
    until = np.minimum(t, end[0]) if end else t
    return np.where(t > start, slope * (until - start), 0.0)

def sd_pulse(height, start, width, t):
    return np.where((t >= start) & (t < start + width), height, 0.0)

## Parameters and initial values

Edit the values below and rerun the notebook from here to try your own scenarios.

In [ ]:
# 1. Model Parameters (Constants)
params = {
    'Customer_Demand': 100,  # units/week
    'Target_Inventory': 400,  # units
    'inventory_adjustment_time': 2,  # week
    'delivery_delay': 4,  # week
    'supply_line_weight': 0  # dimensionless
}

# 2. Initial State (Stocks)
y0 = [
    400,  # Supply_Line [units]
    200  # Inventory [units]
]

## Model equations

`model` returns the rate of change of every stock. It is the system of ordinary differential equations shown above.

In [ ]:
# 3. Model Definition
def model(y, t, p):
    # Unpack stocks
    Supply_Line, Inventory = y

    # Auxiliaries & Flows (dependency order)
    desired_supply_line = (p['Customer_Demand'] * p['delivery_delay'])
    orders = np.maximum(0, ((p['Customer_Demand'] + (((p['Target_Inventory'] - Inventory)) / p['inventory_adjustment_time'])) + ((p['supply_line_weight'] * ((desired_supply_line - Supply_Line))) / p['inventory_adjustment_time'])))
    deliveries = (Supply_Line / p['delivery_delay'])
    sales = p['Customer_Demand']

    # Derivatives (Change in Stocks)
    dSupply_Line_dt = orders - deliveries
    dInventory_dt = deliveries - sales

    return [dSupply_Line_dt, dInventory_dt]

## Simulation

The model runs from t = 0 to t = 80 weeks. The app integrates with RK4 at a fixed
dt = 0.1; this notebook uses SciPy's adaptive `odeint` (LSODA) with tight tolerances, so results agree
with the app within numerical tolerance.

In [ ]:
# 4. Run Simulation
t = np.linspace(0, 80, 801)
sol = odeint(model, y0, t, args=(params,), rtol=1e-8, atol=1e-10)

In [ ]:
# 5. Post-Processing (Calculate variables for plotting)
Supply_Line = sol[:, 0]
Inventory = sol[:, 1]
Customer_Demand = np.full_like(t, params['Customer_Demand'])
Target_Inventory = np.full_like(t, params['Target_Inventory'])
inventory_adjustment_time = np.full_like(t, params['inventory_adjustment_time'])
delivery_delay = np.full_like(t, params['delivery_delay'])
supply_line_weight = np.full_like(t, params['supply_line_weight'])
desired_supply_line = (params['Customer_Demand'] * params['delivery_delay']) + np.zeros_like(t)
orders = np.maximum(0, ((params['Customer_Demand'] + (((params['Target_Inventory'] - Inventory)) / params['inventory_adjustment_time'])) + ((params['supply_line_weight'] * ((desired_supply_line - Supply_Line))) / params['inventory_adjustment_time']))) + np.zeros_like(t)
deliveries = (Supply_Line / params['delivery_delay']) + np.zeros_like(t)
sales = params['Customer_Demand'] + np.zeros_like(t)

## Results

In [ ]:
# 6. Visualization
plt.figure(figsize=(10, 6))
plt.plot(t, Inventory, label='Inventory', linewidth=2)
plt.plot(t, Supply_Line, label='Supply_Line', linewidth=2)

plt.title('System Dynamics Simulation (Python)')
plt.xlabel('Time (week)')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Experiment

Accounting fully for orders already in the supply line: what happens to the overshoot?

The cell below reruns the model with one change and compares each stock against the baseline.

In [ ]:
STOCKS = ['Supply_Line', 'Inventory']

def simulate(param_changes=None, initial_changes=None):
    """Rerun the model with changed parameters and/or initial stock values."""
    p = {**params, **(param_changes or {})}
    y_init = list(y0)
    for stock, value in (initial_changes or {}).items():
        y_init[STOCKS.index(stock)] = value
    return odeint(model, y_init, t, args=(p,), rtol=1e-8, atol=1e-10)

scenario = simulate(param_changes={'supply_line_weight': 1}, initial_changes={})

fig, axes = plt.subplots(len(STOCKS), 1, sharex=True, figsize=(10, 2.8 * len(STOCKS)))
axes = np.atleast_1d(axes)
for i, (ax, stock) in enumerate(zip(axes, STOCKS)):
    ax.plot(t, sol[:, i], label='Baseline', linewidth=2)
    ax.plot(t, scenario[:, i], '--', label='supply_line_weight = 1', linewidth=2)
    ax.set_ylabel(stock)
    ax.grid(True, alpha=0.3)
axes[0].legend(loc='best')
axes[-1].set_xlabel('Time (week)')
fig.suptitle('Baseline vs. scenario')
plt.tight_layout()
plt.show()

## Questions to explore

- Set supply_line_weight to 1. What happens to the overshoot?
- Make delivery_delay longer. Is the oscillation stronger or weaker?